# Lab 3: Data Cleaning & Preprocessing

Welcome to Lab 3! Real-world data is almost never ready to use straight away — it usually has missing values, duplicate rows, inconsistent formatting, wrong data types, and outliers. This lab walks through identifying and fixing each of these problems using Pandas and NumPy.

**After this lab you will be able to:**
- Identify common data quality issues in a real dataset
- Handle missing values using multiple strategies
- Remove duplicate records
- Standardize inconsistent text and date formats
- Fix incorrect data types
- Detect and handle outliers

**Instructions:**
- Write your code between the `### YOUR CODE HERE ###` and `### END ###` markers.
- Run each cell with **Shift + Enter**.
- Compare your output with the **Expected Output** shown below each exercise where provided.

## Why Data Cleaning Matters

Before any model can learn from data, the data itself needs to be trustworthy. Messy data leads directly to wrong conclusions and poor model performance — a model cannot tell the difference between a genuine value and a data-entry mistake unless you clean it first. In most real projects, data cleaning and preprocessing take up more time than model building itself.

The most common problems you will run into are:

- **Missing values** — some cells are simply empty (`NaN`), often because of incomplete data entry.
- **Duplicate records** — the same row appears more than once, which can bias any statistics or model trained on it.
- **Inconsistent formatting** — the same category written in different ways (`"IT"`, `"it"`, `"IT "`), which a computer treats as entirely different values.
- **Incorrect data types** — numbers stored as text (e.g., `"$45,000"` instead of `45000`), which blocks any numeric calculation.
- **Outliers** — values far outside the expected range (e.g., an age of 200), which can distort averages and model training.

We will fix each of these, one at a time, using the dataset below.

## Loading the Dataset

For this lab, you have been given a file named **Messy_Employee_dataset.csv**. It is a small, intentionally messy HR dataset containing exactly the kinds of issues described above.

**Upload the file to this Colab session** using the cell below. When you run it, a button will appear — click it and select `Messy_Employee_dataset.csv` from your computer.

Note: Files uploaded this way only last for the current session. If your Colab runtime disconnects, you will need to re-run this cell and upload the file again.

In [9]:
from google.colab import files

uploaded = files.upload()   # click the button that appears and select Messy_Employee_dataset.csv

Saving Messy_Employee_dataset.csv to Messy_Employee_dataset (1).csv


In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv("Messy_Employee_dataset.csv")
df.head(10)

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False
5,EMP1005,Alice,Garcia,40.0,Sales-Texas,Inactive,6/10/2020,88642.84,alice.garcia@example.com,-5409003485,Good,False
6,EMP1006,Frank,Jones,NaN,Admin-Nevada,Active,4/3/2020,96288.43,frank.jones@example.com,-4518376063,Good,False
7,EMP1007,Bob,Jones,30.0,Cloud Tech-Florida,Inactive,7/17/2022,94497.91,bob.jones@example.com,-4134327559,Average,True
8,EMP1008,Frank,Davis,35.0,Admin-Nevada,Inactive,12/8/2023,115565.82,frank.davis@example.com,-4177656123,Excellent,True
9,EMP1009,Charlie,Johnson,NaN,DevOps-New York,Active,8/4/2022,76561.88,charlie.johnson@example.com,-8156985699,Excellent,True


## Step 1 — Inspecting the Data

Before fixing anything, always inspect the dataset first: how many rows and columns it has, what data types each column has, and how many missing values exist.

In [11]:
df.shape

(1020, 12)

In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   object 
 1   First_Name         1020 non-null   object 
 2   Last_Name          1020 non-null   object 
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   object 
 5   Status             1020 non-null   object 
 6   Join_Date          1020 non-null   object 
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   object 
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   object 
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), object(8)
memory usage: 88.8+ KB


**Exercise:** Print the number of missing values in each column, and separately, the number of fully duplicated rows in the dataset.

In [13]:
### YOUR CODE HERE ###
missing_counts = df.isnull().sum()
duplicate_count = df.duplicated().sum()
### END ###

print(f"Missing values: {missing_counts}")
print(f"Duplicate rows: {duplicate_count}")

Missing values: Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64
Duplicate rows: 0


## Step 2 — Removing Duplicates

Exact duplicate rows add no new information and can skew any summary statistics or model trained on the data.

**Exercise:** Create `df_no_dupes`, a copy of `df` with all fully duplicated rows removed. Keep the first occurrence of each.

In [14]:
### YOUR CODE HERE ###
df_no_dupes = df.drop_duplicates()
### END ###

print(f"Before: {df.shape[0]}, rows | After: {df_no_dupes.shape[0]}, rows")

Before: 1020, rows | After: 1020, rows


## Step 3 — Handling Missing Values

There is no single correct way to handle missing values — the right strategy depends on the column and the situation:

- **Drop rows** when very few rows have missing values and losing them won't hurt the analysis.
- **Fill with the mean/median** for numeric columns, when a reasonable estimate is better than losing the row entirely.
- **Fill with a placeholder** (e.g., `"Unknown"`) for categorical/text columns, when the missing information is itself meaningful.

**Exercise:** On `df_no_dupes`, fill missing `Age` values with the column's median, fill missing `Salary` values with the column's mean, and drop any remaining rows where `Email` is still missing. Store the result in `df_clean`.

In [15]:
df_clean = df_no_dupes.copy()

### YOUR CODE HERE ###
df_clean["Age"] = df_clean["Age"].fillna(df_clean["Age"].median())
df_clean["Salary"] = df_clean["Salary"].fillna(df_clean["Salary"].mean())
df_clean = df_clean.dropna(subset=["Email"])
### END ###

print(df_clean.isnull().sum())

Employee_ID          0
First_Name           0
Last_Name            0
Age                  0
Department_Region    0
Status               0
Join_Date            0
Salary               0
Email                0
Phone                0
Performance_Score    0
Remote_Work          0
dtype: int64


## Step 4 — Standardizing Inconsistent Text

Look closely at the `First_Name`, `Last_Name` and `Department_Region` columns — the same value often appears with different capitalization or extra whitespace (e.g., `"bilal ahmed"` vs `"BILAL AHMED"`, `"Cloud "` with a trailing space). To a computer, these are different strings unless you standardize them.

**Exercise:** On `df_clean`, strip extra whitespace and convert `First_Name` and `Last_Name` to title case, and strip whitespace and convert `Department_Regions` to title case as well.

In [18]:
### YOUR CODE HERE ###
df_clean["First_Name"] = df_clean["First_Name"].str.strip().str.title()
df_clean["Last_Name"] = df_clean["Last_Name"].str.strip().str.title()
df_clean[["Department", "Region"]] = df_clean["Department_Region"].str.split("-", n=1, expand=True)
df_clean["Department"] = df_clean["Department"].str.strip().str.title()
df_clean["Region"] = df_clean["Region"].str.strip().str.title()
### END ###

print(df_clean["Department"].unique())

['Devops' 'Finance' 'Admin' 'Cloud Tech' 'Sales' 'Hr']


**Expected Output (Department values, order may vary):**
```
['Devops' 'Finance' 'Admin' 'Cloud Tech' 'Sales' 'Hr']
```

## Step 5 — Fixing Incorrect Data Types

The `Phone` column has negative long integers like `"-1651623197"`.

**Exercise:** Write a function `clean_phone(value)` that removes `-` character(if present), replaces it with `-` character and converts the result to a str, and use `.apply()` to create a cleaned `Phone` column in `df_clean`.

In [24]:
def clean_phone(value):
        ### YOUR CODE HERE ###
        value = value.replace("-", "+")

        return str(value)
        ### END ###
        pass

df_clean["Phone"] = df_clean["Phone"].apply(clean_phone)
print(df_clean["Phone"].dtype)
print(df_clean["Phone"].head(10))

object
0    +1651623197
1    +1898471390
2    +5596363211
3    +3476490784
4    +1586734256
5    +5409003485
6    +4518376063
7    +4134327559
8    +4177656123
9    +8156985699
Name: Phone, dtype: object


## Standardizing Dates

The `Join_Date` column has several different date formats (`"2021-03-15"`, `"15/06/2020"`, `"March 3, 2022"`). Pandas can parse most common formats automatically with `pd.to_datetime()` using `errors="coerce"`, which turns anything it cannot parse into `NaT` (a missing date) instead of crashing.

In [25]:
df_clean["Join_Date"] = pd.to_datetime(df_clean["Join_Date"], errors="coerce")
df_clean["Join_Date"].head(10)

,Join_Date
0,2021-04-02
1,2020-07-10
2,2023-12-07
3,2021-11-27
4,2022-01-05
5,2020-06-10
6,2020-04-03
7,2022-07-17
8,2023-12-08
9,2022-08-04


## Step 6 — Detecting Outliers

An outlier is a value far outside the range you would reasonably expect — for example, an employee age of 200. One common way to detect outliers is the IQR (Interquartile Range) method: values below `Q1 - 1.5*IQR` or above `Q3 + 1.5*IQR` are flagged.

**Exercise:** Using the IQR method on the `Age` column, identify which rows are outliers and store them in `age_outliers`. Then replace outlier ages with the column's median.

In [26]:
Q1 = df_clean["Age"].quantile(0.25)
Q3 = df_clean["Age"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

### YOUR CODE HERE ###
age_outliers = df_clean[(df_clean["Age"] < lower_bound) | (df_clean["Age"] > upper_bound)]
median_age = df_clean["Age"].median()
df_clean.loc[(df_clean["Age"] < lower_bound) | (df_clean["Age"] > upper_bound), "Age"] = median_age
df_clean["Age"].describe()
### END ###

print(age_outliers[["Employee_ID", "Age"]])

Empty DataFrame
Columns: [Employee_ID, Age]
Index: []


**What to remember from this lab:**
- Always inspect a dataset (`.info()`, `.isnull().sum()`, `.duplicated().sum()`) before doing anything else
- There is no single "correct" way to handle missing values — the right strategy depends on context
- Inconsistent text formatting must be standardized before grouping or filtering by it
- Data types must be correct before any numeric operation will work
- The IQR method is one simple, common way to flag outliers

## Lab Tasks

Complete the following tasks in this notebook, below this cell, using `df_clean` (or the original `df` where specified).

1. **Full Pipeline Recap:** In a markdown cell, write 3-4 sentences summarizing, in your own words, the full cleaning pipeline you just performed on this dataset (duplicates → missing values → text → types → dates → outliers).
2. **Salary Outliers:** Repeat the IQR outlier-detection method from Step 6, but this time apply it to the cleaned `Salary` column. Print any rows flagged as outliers.
3. **Department Summary:** Using `df_clean`, compute the average `Salary` and average `Age` per `Department`, in a single `.groupby().agg()` call.
4. **Export the Cleaned Data:** Save `df_clean` to a new file named `employee_records_clean.csv` using `.to_csv()`, without the index column.
5. **Reflection:** In a markdown cell, note which cleaning step you found most challenging and why.

In [ ]:
# Task 1 (markdown summary can go in a text cell below, or as a comment here)


In this lab, I performed data cleaning on the provided messy employee dataset through several important steps. First, I inspected the dataset to understand its structure, column data types, missing values and duplicate records. Then I removed duplicate rows (though none were found). After that, I handled missing values by filling the `Age` column with its median and the `Salary` column with its mean. I also standardized text columns by stripping whitespace characters and converting names to title case, and I split the `Department_Region` column into two separate columns: `Department` and `Region`. I fixed the data type of the `Phone` column by converting negative integers into proper string format starting with a `+` sign. Finally, I converted the `Join_Date` column into proper datetime format and detected outliers in the `Age` column using the IQR method (though no outliers were found).

In [27]:
# Task 2
Q1 = df_clean["Salary"].quantile(0.25)
Q3 = df_clean["Salary"].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

salary_outliers = df_clean[(df_clean["Salary"] < lower_bound) | (df_clean["Salary"] > upper_bound)]
median_salary = df_clean["Salary"].median()
df_clean.loc[(df_clean["Salary"] < lower_bound) | (df_clean["Salary"] > upper_bound), "Salary"] = median_salary
df_clean["Salary"].describe()

print(salary_outliers[["Employee_ID", "Salary"]])

Empty DataFrame
Columns: [Employee_ID, Salary]
Index: []


In [29]:
# Task 3
employee_summary = df_clean.groupby("Department").agg(
    Average_Salary = ("Salary", "mean"),
    Average_Age = ("Age", "mean")
).round(2)

print(employee_summary)

            Average_Salary  Average_Age
Department                             
Admin             85190.81        31.75
Cloud Tech        84930.98        31.68
Devops            85993.25        31.93
Finance           82274.90        32.59
Hr                85466.19        31.87
Sales             86867.33        31.97


In [30]:
# Task 4
df_clean.to_csv("employee_records_clean.csv", index=False)

print("Cleaned data exported successfully as 'employee_records_clean.csv'")

Cleaned data exported successfully as 'employee_records_clean.csv'


_Write your Task 5 reflection here._

The cleaning step I found most challenging was **standardizing inconsistent text**. Although the data looked mostly clean at first glance, I had to carefully strip whitespace and convert text to title case for columns like `First_Name` and `Last_Name`. The complex part was splitting the `Department_Region` column into two separate columns (`Department` and `Region`) while also cleaning the text properly to enable the cleaned dataset to display less ambiguous, more accurate and comprehensive data. It required extra attention to ensure the data values remained consistent and correctly formatted.

In [31]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/numair-2003/AIML-Internship-NumairFahad.git

Mounted at /content/drive
Cloning into 'AIML-Internship-NumairFahad'...
remote: Enumerating objects: 12, done.
remote: Counting objects: 100% (12/12), done.
remote: Compressing objects: 100% (10/10), done.
remote: Total 12 (delta 1), reused 3 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (12/12), 14.50 KiB | 530.00 KiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
!cp "/content/drive/MyDrive/Colab Notebooks/YourNotebook.ipynb" "/content/your-repo/"